# Figure 4

Import statements

In [1]:
from datetime import datetime

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, weibull_min

## Filenames

In [2]:
ts_or_td = "TD"

start = 1940
stop = 2023
nens = 10

namelist_f = [
    [
        f"/glade/work/smhenry/NeuralGCM/data/tracks/factual/ens{X}_{YYYY}_JASO_TC_tracks_factual.txt"
        for X in range(1, nens + 1)
    ]
    for YYYY in range(start, stop + 1)
]

## Counting functions

In [3]:
def count_NA(file):
    """
    counts the number of TCs from TempestExtremes output in the North Atlantic Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NA = False

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if is_in_NA:
                    count += 1
                is_in_NA = False
                in_TC = True
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if (
                        (285 <= lon <= 360 and 0 <= lat <= 50)
                        or (276 <= lon < 285 and 10 <= lat <= 50)
                        or (262 <= lon < 276 and 16.5 <= lat <= 50)
                    ):
                        is_in_NA = True

        if is_in_NA:
            count += 1

    return count


def count_NWP(file):
    """
    counts the number of TCs from TempestExtremes output in the Northwest Pacific Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NWP = False

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if is_in_NWP:
                    count += 1
                is_in_NWP = False
                in_TC = True
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if 100 <= lon <= 180 and 0 <= lat <= 50:  # need to update
                        is_in_NWP = True

        if is_in_NWP:
            count += 1

    return count


def count_NEP(file):
    """
    counts the number of TCs from TempestExtremes output in the Northeast Pacific Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NEP = False

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if is_in_NEP:
                    count += 1
                is_in_NEP = False
                in_TC = True
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if (
                        (275 <= lon <= 284 and 0 <= lat <= 10)
                        or (260 <= lon < 275 and 0 <= lat <= 16.5)
                        or (180 <= lon < 260 and 0 <= lat <= 50)
                    ):
                        is_in_NEP = True

        if is_in_NEP:
            count += 1

    return count


def count_TCs_ibtracs(year, basin, type):  # working yay!
    """
    returns the count of TCs from IBTrACS in a given basin for a given year
    input:
        year (int): desired year
        basin (str): desired basin
            keys: "AL", "WP", "EP"
        type (str): storm type
            keys: "TD" (tropical depression), "TS" (tropical storm), "HR" or "TY" (hurricane or typhoon, basin dependent)
            also see (DB: disturbance, TD: tropical depression, TS: tropical storm, TY: typhoon, ST: super typhoon, TC: tropical cyclone,
                    HU,HR: hurricane, SD: subtropical depression, SS: subtropical storm, EX: extratropical systems, PT: post tropical,
                    IN: inland, DS: dissipating, LO: low, WV: tropical wave, ET: extrapolated, MD: monsoon depression, XX: unknown)
    output:
        count (int): count of TCS
    criteria:
        TCs must last at least 54 hours to be consistent with TempestExtremes tracking
    """

    # open dataframe
    df = pd.read_csv(
        f"/glade/work/smhenry/NeuralGCM/data/ibtracs/IBTrACS_{year}_JASO.csv"
    )
    df["time"] = pd.to_datetime(df["time"])

    # Calculate storm durations
    storm_durations = df.groupby("stormid")["time"].agg(
        lambda x: (x.max() - x.min()).total_seconds() / 3600
    )

    # Keep only storms lasting at least 54 hours
    valid_storms = storm_durations[storm_durations >= 54].index
    df = df[df["stormid"].isin(valid_storms) & df["stormid"].str.contains(basin)]

    # Filter by storm type
    type_dict = {
        "TD": "HU|HR|TY|ST|TS|TC|SS|TD|SD",
        "TS": "HU|HR|TY|ST|TS|TC|SS",
        "HU": "HU|HR|TY|ST",
        "TY": "HU|HR|TY|ST",
    }

    if type in type_dict:
        df = df[df["type"].str.contains(type_dict[type])]

    return df["stormid"].nunique()

In [4]:
def count_NA_mo(file, mo):
    """
    counts the number of TCs from TempestExtremes output in the North Atlantic Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NA = False
    in_mo = False

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if (
                    is_in_NA and in_mo
                ):  # this addresses the last TC, then bools are reset
                    count += 1
                is_in_NA = False
                in_TC = True
                if line.split()[3] == mo:
                    in_mo = True
                else:
                    in_mo = False
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if (
                        (285 <= lon <= 360 and 0 <= lat <= 50)
                        or (276 <= lon < 285 and 10 <= lat <= 50)
                        or (262 <= lon < 276 and 16.5 <= lat <= 50)
                    ):
                        is_in_NA = True

        if is_in_NA and in_mo:
            count += 1

    return count


def count_NWP_mo(file, mo):
    """
    counts the number of TCs from TempestExtremes output in the Northwest Pacific Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NWP = False
    in_mo = None

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if (
                    is_in_NWP and in_mo
                ):  # this addresses the last TC, then bools are reset
                    count += 1
                is_in_NWP = False
                in_TC = True
                if line.split()[3] == mo:
                    in_mo = True
                else:
                    in_mo = False
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if 100 <= lon <= 180 and 0 <= lat <= 50:
                        is_in_NWP = True

        if is_in_NWP and in_mo:
            count += 1

    return count


def count_NEP_mo(file, mo):
    """
    counts the number of TCs from TempestExtremes output in the Northeast Pacific Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NEP = False
    in_mo = None

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if (
                    is_in_NEP and in_mo
                ):  # this addresses the last TC, then bools are reset
                    count += 1
                is_in_NEP = False
                in_TC = True
                if line.split()[3] == mo:
                    in_mo = True
                else:
                    in_mo = False
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if (
                        (275 <= lon <= 284 and 0 <= lat <= 10)
                        or (260 <= lon < 275 and 0 <= lat <= 16.5)
                        or (180 <= lon < 260 and 0 <= lat <= 50)
                    ):
                        is_in_NEP = True

        if is_in_NEP and in_mo:
            count += 1

    return count


def count_TCs_ibtracs_mo(
    year, mo, basin, type
):  # updated to filter by storm start month
    """
    Returns the count of TCs from IBTrACS in a given basin for a specific year and month (based on storm start month).

    Inputs:
        year (int): desired year
        mo (int): desired month (1–12)
        basin (str): desired basin
            keys: "AL", "WP", "EP"
        type (str): storm type
            keys: "TD" (tropical depression), "TS" (tropical storm), "HR" or "TY" (hurricane or typhoon)
    Output:
        count (int): count of valid tropical cyclones (TCs)

    Criteria:
        - Storms must last at least 54 hours
        - Storm must start in the given month
    """
    # open dataframe
    df = pd.read_csv(
        f"/glade/work/smhenry/NeuralGCM/data/ibtracs/IBTrACS_{year}_JASO.csv"
    )

    df["time"] = pd.to_datetime(df["time"])

    # Calculate storm durations
    storm_durations = df.groupby("stormid")["time"].agg(
        lambda x: (x.max() - x.min()).total_seconds() / 3600
    )

    # Keep only storms lasting at least 54 hours
    valid_storms = storm_durations[storm_durations >= 54].index
    df_valid = df[df["stormid"].isin(valid_storms) & df["stormid"].str.contains(basin)]

    # Filter by storm type
    type_dict = {
        "TD": "HU|HR|TY|ST|TS|TC|SS|TD|SD",
        "TS": "HU|HR|TY|ST|TS|TC|SS",
        "HU": "HU|HR|TY|ST",
        "TY": "HU|HR|TY|ST",
    }

    if type in type_dict:
        df_type = df_valid[df_valid["type"].str.contains(type_dict[type])]

    # Get the first timestamp per storm
    storm_start_month = df_type.groupby("stormid")["time"].min().dt.month
    storms_starting_in_month = storm_start_month[storm_start_month == mo].index

    df_return = df_type[df_type["stormid"].isin(storms_starting_in_month)]

    return df_return["stormid"].nunique()

## Processing functions

In [5]:
def process_counts(namelist, basin, start, exclude=None, ens=False, min_max=False):
    """
    inputs:
        namelist: list of strs of filenames
        basin: str, "NA", "NWP", "NEP"
        start: start year of simulation data analysis
        exclude: list of sets of [[yr, ens]] to exclude
        ens: bool, contains ens data or not
        min_max: bool, return min max data for each year or not

    returns:
        ensmean_count: list of yearly ensemble mean TC counts, or if ens=False, just the yearly counts
        twenty: 20th percentile for each year, None if ens=False
        eighty: 80th percentile for each year, None if ens=False
        min: yearly min TC count, None if ens=False
        max: yearly max TC count, None if ens=False
    """

    nyr = np.shape(namelist)[0]
    if ens:
        nens = np.shape(namelist)[1]
    else:
        nens = 1

    # initalize array of TC counts for storage
    if ens:
        counts = np.zeros((nyr, nens))
    else:
        counts = np.zeros((nyr, 1))

    # count
    for iyr in range(nyr):
        if ens:
            for iens in range(nens):
                if exclude:
                    for i in range(len(exclude)):
                        if iyr == (exclude[i][0] - start) and iens == (
                            exclude[i][1] - 1
                        ):
                            counts[iyr][iens] = np.nan

                        else:
                            if basin == "NA":
                                counts[iyr][iens] = count_NA(namelist[iyr][iens])
                            elif basin == "NWP":
                                counts[iyr][iens] = count_NWP(namelist[iyr][iens])
                            elif basin == "NEP":
                                counts[iyr][iens] = count_NEP(namelist[iyr][iens])
                            else:
                                counts[iyr][iens] = np.nan

                else:
                    if basin == "NA":
                        counts[iyr][iens] = count_NA(namelist[iyr][iens])
                    elif basin == "NWP":
                        counts[iyr][iens] = count_NWP(namelist[iyr][iens])
                    elif basin == "NEP":
                        counts[iyr][iens] = count_NEP(namelist[iyr][iens])
                    else:
                        counts[iyr][iens] = np.nan

        else:
            if basin == "NA":
                counts[iyr] = count_NA(namelist[iyr])
            elif basin == "NWP":
                counts[iyr] = count_NWP(namelist[iyr])
            elif basin == "NEP":
                counts[iyr] = count_NEP(namelist[iyr])
            else:
                counts[iyr] = np.nan

    # take the ensemble mean and get a 20th and 80th percentile spread

    if ens:
        # initalize arrays for storage
        count = np.zeros((nyr, 1))
        twenty = np.zeros((nyr, 1))
        eighty = np.zeros((nyr, 1))
        min = np.zeros((nyr, 1))
        max = np.zeros((nyr, 1))

        for iyr in range(nyr):
            # count for that year
            yr_count = counts[iyr][:]

            # take mean across year
            count[iyr] = np.nanmean(yr_count)

            # take the 20th and 80th percentiles
            twenty[iyr] = np.nanpercentile(yr_count, 20)
            eighty[iyr] = np.nanpercentile(yr_count, 80)

            # take the mins and maxs
            if min_max:
                min[iyr] = np.nanmin(yr_count)
                max[iyr] = np.nanmax(yr_count)
            else:
                min = None
                max = None

    else:
        # initalize arrays for storage
        count = np.zeros((nyr, 1))
        twenty = np.zeros((nyr, 1))
        eighty = np.zeros((nyr, 1))
        min = np.zeros((nyr, 1))
        max = np.zeros((nyr, 1))

        for iyr in range(nyr):
            # count for that year
            yr_count = counts[iyr][:]

            # take mean across year
            count[iyr] = np.nanmean(yr_count)

            # take the 20th and 80th percentiles
            twenty[iyr] = np.nanpercentile(yr_count, 20)
            eighty[iyr] = np.nanpercentile(yr_count, 80)

            # take the mins and maxs
            if min_max:
                min[iyr] = np.nanmin(yr_count)
                max[iyr] = np.nanmax(yr_count)
            else:
                min = None
                max = None

    return count, twenty, eighty, min, max


def process_counts_mo(namelist, basin, start, mo, exclude=None, nens=1, min_max=False):
    """
    inputs:
        namelist: list of strs of filenames
        basin: str, "NA", "NWP", "NEP"
        start: start year of simulation data analysis
        exclude: list of sets of [[yr, ens]] to exclude
        ens: bool, contains ens data or not
        min_max: bool, return min max data for each year or not

    returns:
        ensmean_count: list of yearly ensemble mean TC counts, or if ens=False, just the yearly counts
        twenty: 20th percentile for each year, None if ens=False
        eighty: 80th percentile for each year, None if ens=False
        min: yearly min TC count, None if ens=False
        max: yearly max TC count, None if ens=False
    """

    nyr = np.shape(namelist)[0]

    # initalize array of TC counts for storage
    if nens > 1:
        counts = np.zeros((nyr, nens))
    else:
        counts = np.zeros((nyr, 1))

    # count
    for iyr in range(nyr):
        if nens > 1:
            for iens in range(nens):
                if exclude:
                    for i in range(len(exclude)):
                        if iyr == (exclude[i][0] - start) and iens == (
                            exclude[i][1] - 1
                        ):
                            counts[iyr][iens] = np.nan

                        else:
                            if basin == "NA":
                                counts[iyr][iens] = count_NA_mo(namelist[iyr][iens], mo)
                            elif basin == "NWP":
                                counts[iyr][iens] = count_NWP_mo(
                                    namelist[iyr][iens], mo
                                )
                            elif basin == "NEP":
                                counts[iyr][iens] = count_NEP_mo(
                                    namelist[iyr][iens], mo
                                )
                            else:
                                counts[iyr][iens] = np.nan

                else:
                    if basin == "NA":
                        counts[iyr][iens] = count_NA_mo(namelist[iyr][iens], mo)
                    elif basin == "NWP":
                        counts[iyr][iens] = count_NWP_mo(namelist[iyr][iens], mo)
                    elif basin == "NEP":
                        counts[iyr][iens] = count_NEP_mo(namelist[iyr][iens], mo)
                    else:
                        counts[iyr][iens] = np.nan

        else:
            if basin == "NA":
                counts[iyr][iens] = count_NA_mo(namelist[iyr][iens], mo)
            elif basin == "NWP":
                counts[iyr][iens] = count_NWP_mo(namelist[iyr][iens], mo)
            elif basin == "NEP":
                counts[iyr][iens] = count_NEP_mo(namelist[iyr][iens], mo)
            else:
                counts[iyr][iens] = np.nan

    # take the ensemble mean and get a 20th and 80th percentile spread

    if nens > 1:
        # initalize arrays for storage
        count = np.zeros((nyr, 1))
        twenty = np.zeros((nyr, 1))
        eighty = np.zeros((nyr, 1))
        min = np.zeros((nyr, 1))
        max = np.zeros((nyr, 1))

        for iyr in range(nyr):
            # count for that year
            yr_count = counts[iyr][:]

            # take mean across year
            count[iyr] = np.nanmean(yr_count)

            # take the 20th and 80th percentiles
            twenty[iyr] = np.nanpercentile(yr_count, 20)
            eighty[iyr] = np.nanpercentile(yr_count, 80)

            # take the mins and maxs
            if min_max:
                min[iyr] = np.nanmin(yr_count)
                max[iyr] = np.nanmax(yr_count)
            else:
                min = None
                max = None

    else:
        # initalize arrays for storage
        count = np.zeros((nyr, 1))
        twenty = np.zeros((nyr, 1))
        eighty = np.zeros((nyr, 1))
        min = np.zeros((nyr, 1))
        max = np.zeros((nyr, 1))

        for iyr in range(nyr):
            # count for that year
            yr_count = counts[iyr][:]

            # take mean across year
            count[iyr] = np.nanmean(yr_count)

            # take the 20th and 80th percentiles
            twenty[iyr] = np.nanpercentile(yr_count, 20)
            eighty[iyr] = np.nanpercentile(yr_count, 80)

            # take the mins and maxs
            if min_max:
                min[iyr] = np.nanmin(yr_count)
                max[iyr] = np.nanmax(yr_count)
            else:
                min = None
                max = None

    return count, twenty, eighty, min, max

## Lifetime function

In [6]:
def get_lifetime_NA(file):
    """
    returns a list of the TC lifetime of each track in file
    input: file, a file name which is an output from TempestExtremes
    output: lifetimes, a list of the lifetime [hours] of each track in file
        length of lifetimes should be equal to count
    """

    rows = []
    current_tc_id = None
    tc_tracks = {}

    with open(file, "r") as f:
        for line in f:
            if line.startswith("start"):
                current_tc_id = len(tc_tracks)
                tc_tracks[current_tc_id] = []
            else:
                data = line.split()
                if len(data) >= 10:
                    lon = float(data[2])
                    lat = float(data[3])
                    year, month, day, hour = map(int, data[6:10])
                    timestamp = datetime(year, month, day, hour)
                    tc_tracks[current_tc_id].append((lon, lat, timestamp))

    lifetimes = []
    for tc_id, track in tc_tracks.items():
        if not track:
            continue

        first_lon, first_lat, _ = track[0]
        if (
            ((first_lon >= 285) & (first_lon <= 360) & (first_lat >= 0))
            | ((first_lon >= 276) & (first_lon <= 285) & (first_lat >= 10))
            | ((first_lon >= 262) & (first_lon <= 276) & (first_lat >= 16.5))
        ):
            start_time = track[0][2]
            end_time = track[-1][2]
            lifetime_days = (end_time - start_time).total_seconds() / 3600 / 24
            lifetimes.append(int(lifetime_days))

    return lifetimes


def get_lifetime_NEP(file):
    """
    returns a list of the TC lifetime of each track in file
    input: file, a file name which is an output from TempestExtremes
    output: lifetimes, a list of the lifetime [hours] of each track in file
        length of lifetimes should be equal to count
    """

    rows = []
    current_tc_id = None
    tc_tracks = {}

    with open(file, "r") as f:
        for line in f:
            if line.startswith("start"):
                current_tc_id = len(tc_tracks)
                tc_tracks[current_tc_id] = []
            else:
                data = line.split()
                if len(data) >= 10:
                    lon = float(data[2])
                    lat = float(data[3])
                    year, month, day, hour = map(int, data[6:10])
                    timestamp = datetime(year, month, day, hour)
                    tc_tracks[current_tc_id].append((lon, lat, timestamp))

    lifetimes = []
    for tc_id, track in tc_tracks.items():
        if not track:
            continue

        first_lon, first_lat, _ = track[0]
        if (
            ((first_lon >= 275) & (first_lon <= 284) & (first_lat <= 10))
            | ((first_lon >= 260) & (first_lon <= 275) & (first_lat <= 16.5))
            | ((first_lon >= 180) & (first_lon <= 260) & (first_lat >= 0))
        ):
            start_time = track[0][2]
            end_time = track[-1][2]
            lifetime_days = (end_time - start_time).total_seconds() / 3600 / 24
            lifetimes.append(int(lifetime_days))

    return lifetimes


def get_lifetime_NWP(file):
    """
    returns a list of the TC lifetime of each track in file
    input: file, a file name which is an output from TempestExtremes
    output: lifetimes, a list of the lifetime [hours] of each track in file
        length of lifetimes should be equal to count
    """

    rows = []
    current_tc_id = None
    tc_tracks = {}

    with open(file, "r") as f:
        for line in f:
            if line.startswith("start"):
                current_tc_id = len(tc_tracks)
                tc_tracks[current_tc_id] = []
            else:
                data = line.split()
                if len(data) >= 10:
                    lon = float(data[2])
                    lat = float(data[3])
                    year, month, day, hour = map(int, data[6:10])
                    timestamp = datetime(year, month, day, hour)
                    tc_tracks[current_tc_id].append((lon, lat, timestamp))

    lifetimes = []
    for tc_id, track in tc_tracks.items():
        if not track:
            continue

        first_lon, first_lat, _ = track[0]
        if (first_lon >= 100) & (first_lon <= 180) & (first_lat >= 0):
            start_time = track[0][2]
            end_time = track[-1][2]
            lifetime_days = (end_time - start_time).total_seconds() / 3600 / 24
            lifetimes.append(int(lifetime_days))

    return lifetimes


def get_lifetime_IBTrACS(year, basin, type):
    # open dataframe
    df = pd.read_csv(
        f"/glade/work/smhenry/NeuralGCM/data/ibtracs/IBTrACS_{year}_JASO.csv"
    )

    df["time"] = pd.to_datetime(df["time"])

    # Calculate storm durations
    storm_durations = df.groupby("stormid")["time"].agg(
        lambda x: (x.max() - x.min()).total_seconds() / 3600
    )

    # Keep only storms lasting at least 54 hours
    valid_storms = storm_durations[storm_durations >= 54].index
    df_valid = df[df["stormid"].isin(valid_storms) & df["stormid"].str.contains(basin)]

    # Filter by storm type
    type_dict = {
        ts_or_td: "HU|HR|TY|ST|TS|TC|SS|TD|SD",
        "TS": "HU|HR|TY|ST|TS|TC|SS",
        "HU": "HU|HR|TY|ST",
        "TY": "HU|HR|TY|ST",
    }

    if type in type_dict:
        df_type = df_valid[df_valid["type"].str.contains(type_dict[type])]

    # Lifetimes
    lifetimes = df_type.groupby("stormid")["time"].agg(
        lambda x: (x.max() - x.min()).total_seconds() / 3600 / 24
    )

    lifetime_values = lifetimes.values
    if len(lifetime_values) > 0:
        lifetime_values = lifetime_values[np.where(lifetime_values > (54 / 24))]

    return lifetime_values


values = np.arange(54 / 24, 24 + 54 / 24 + 1 / 24, 1 / 24)

## Process data

In [ ]:
f_exclude = [
    [1992, 3],
    [2005, 17],
    [2012, 14],
    [2017, 3],
    [2017, 12],
    [2017, 13],
    [2021, 16],
]

# north atlantic
f_ensmean_count_NA, f_twenty_NA, f_eighty_NA, f_min_NA, f_max_NA = process_counts(
    namelist_f, "NA", start, ens=True, min_max=True, exclude=f_exclude
)
# northwest pacific
f_ensmean_count_NWP, f_twenty_NWP, f_eighty_NWP, f_min_NWP, f_max_NWP = process_counts(
    namelist_f, "NWP", start, ens=True, min_max=True, exclude=f_exclude
)
# northeast pacific
f_ensmean_count_NEP, f_twenty_NEP, f_eighty_NEP, f_min_NEP, f_max_NEP = process_counts(
    namelist_f, "NEP", start, ens=True, min_max=True, exclude=f_exclude
)

In [ ]:
p1_start = 1949
p1_stop = 1985
p2_start = 1987
p2_stop = 2023

years = np.arange(start, stop + 1, 1)

i_p1_start = np.where(years == p1_start)[0].item()
i_p1_stop = np.where(years == p1_stop)[0].item()
i_p2_start = np.where(years == p2_start)[0].item()
i_p2_stop = np.where(years == p2_stop)[0].item()

mo_list = ["7", "8", "9", "10"]

mo_count_NA_p1 = []
mo_count_NWP_p1 = []
mo_count_NEP_p1 = []

mo_count_NA_p2 = []
mo_count_NWP_p2 = []
mo_count_NEP_p2 = []

for mo in mo_list:
    # north atlantic
    this_mo_count_NA, twenty_NA, eighty_NA, min_NA, max_NA = process_counts_mo(
        namelist_f[i_p1_start:i_p1_stop][:],
        "NA",
        start,
        mo,
        nens=10,
        min_max=False,
        exclude=f_exclude,
    )
    this_mo_count_NA = (
        this_mo_count_NA.flatten() / f_ensmean_count_NA.flatten()[i_p1_start:i_p1_stop]
    )

    # northwest pacific
    this_mo_count_NWP, twenty_NWP, eighty_NWP, min_NWP, max_NWP = process_counts_mo(
        namelist_f[i_p1_start:i_p1_stop][:],
        "NWP",
        start,
        mo,
        nens=10,
        min_max=False,
        exclude=f_exclude,
    )
    this_mo_count_NWP = (
        this_mo_count_NWP.flatten()
        / f_ensmean_count_NWP.flatten()[i_p1_start:i_p1_stop]
    )

    # northeast pacific
    this_mo_count_NEP, twenty_NEP, eighty_NEP, min_NEP, max_NEP = process_counts_mo(
        namelist_f[i_p1_start:i_p1_stop][:],
        "NEP",
        start,
        mo,
        nens=10,
        min_max=False,
        exclude=f_exclude,
    )
    this_mo_count_NEP = (
        this_mo_count_NEP.flatten()
        / f_ensmean_count_NEP.flatten()[i_p1_start:i_p1_stop]
    )

    mo_count_NA_p1.append(this_mo_count_NA.flatten())
    mo_count_NWP_p1.append(this_mo_count_NWP.flatten())
    mo_count_NEP_p1.append(this_mo_count_NEP.flatten())

for mo in mo_list:
    # north atlantic
    this_mo_count_NA, twenty_NA, eighty_NA, min_NA, max_NA = process_counts_mo(
        namelist_f[i_p2_start:i_p2_stop][:],
        "NA",
        start,
        mo,
        nens=10,
        min_max=False,
        exclude=f_exclude,
    )
    this_mo_count_NA = (
        this_mo_count_NA.flatten() / f_ensmean_count_NA.flatten()[i_p2_start:i_p2_stop]
    )

    # northwest pacific
    this_mo_count_NWP, twenty_NWP, eighty_NWP, min_NWP, max_NWP = process_counts_mo(
        namelist_f[i_p2_start:i_p2_stop][:],
        "NWP",
        start,
        mo,
        nens=10,
        min_max=False,
        exclude=f_exclude,
    )
    this_mo_count_NWP = (
        this_mo_count_NWP.flatten()
        / f_ensmean_count_NWP.flatten()[i_p2_start:i_p2_stop]
    )

    # northeast pacific
    this_mo_count_NEP, twenty_NEP, eighty_NEP, min_NEP, max_NEP = process_counts_mo(
        namelist_f[i_p2_start:i_p2_stop][:],
        "NEP",
        start,
        mo,
        nens=10,
        min_max=False,
        exclude=f_exclude,
    )
    this_mo_count_NEP = (
        this_mo_count_NEP.flatten()
        / f_ensmean_count_NEP.flatten()[i_p2_start:i_p2_stop]
    )

    mo_count_NA_p2.append(this_mo_count_NA.flatten())
    mo_count_NWP_p2.append(this_mo_count_NWP.flatten())
    mo_count_NEP_p2.append(this_mo_count_NEP.flatten())

In [ ]:
obs_TC_count_NA = []
obs_HR_count_NA = []
obs_TC_count_NEP = []
obs_TY_count_NEP = []
obs_TC_count_NWP = []
obs_TY_count_NWP = []

for year in range(start, stop + 1):
    if year < 1949:
        if year < 1945:
            obs_TC_count_NWP.append(np.nan)
            obs_TY_count_NWP.append(np.nan)
        else:
            obs_TC_count_NWP.append(count_TCs_ibtracs(year, "WP", ts_or_td))
            obs_TY_count_NWP.append(count_TCs_ibtracs(year, "WP", "TY"))

        obs_TC_count_NEP.append(np.nan)
        obs_TY_count_NEP.append(np.nan)
    else:
        obs_TC_count_NEP.append(count_TCs_ibtracs(year, "EP", ts_or_td))
        obs_TY_count_NEP.append(count_TCs_ibtracs(year, "EP", "TY"))
        obs_TC_count_NWP.append(count_TCs_ibtracs(year, "WP", ts_or_td))
        obs_TY_count_NWP.append(count_TCs_ibtracs(year, "WP", "TY"))

    obs_TC_count_NA.append(count_TCs_ibtracs(year, "AL", ts_or_td))
    obs_HR_count_NA.append(count_TCs_ibtracs(year, "AL", "HU"))

obs_mo_count_NA_p1 = [[], [], [], []]
obs_mo_count_NEP_p1 = [[], [], [], []]
obs_mo_count_NWP_p1 = [[], [], [], []]

obs_mo_count_NA_p2 = [[], [], [], []]
obs_mo_count_NEP_p2 = [[], [], [], []]
obs_mo_count_NWP_p2 = [[], [], [], []]

obs_lifetimes_NA = []
obs_lifetimes_NEP = []
obs_lifetimes_NWP = []

mo_list = [7, 8, 9, 10]

for year in range(start, stop + 1):
    for i, mo in enumerate(mo_list):
        if p1_start <= year <= p1_stop:
            obs_mo_count_NA_p1[i].append(count_TCs_ibtracs_mo(year, mo, "AL", ts_or_td))
            obs_mo_count_NEP_p1[i].append(
                count_TCs_ibtracs_mo(year, mo, "EP", ts_or_td)
            )
            obs_mo_count_NWP_p1[i].append(
                count_TCs_ibtracs_mo(year, mo, "WP", ts_or_td)
            )
        elif p2_start <= year <= p2_stop:
            obs_mo_count_NA_p2[i].append(count_TCs_ibtracs_mo(year, mo, "AL", ts_or_td))
            obs_mo_count_NEP_p2[i].append(
                count_TCs_ibtracs_mo(year, mo, "EP", ts_or_td)
            )
            obs_mo_count_NWP_p2[i].append(
                count_TCs_ibtracs_mo(year, mo, "WP", ts_or_td)
            )

    obs_lifetimes_NA.extend(get_lifetime_IBTrACS(year, "AL", ts_or_td))
    obs_lifetimes_NEP.extend(get_lifetime_IBTrACS(year, "EP", ts_or_td))
    obs_lifetimes_NWP.extend(get_lifetime_IBTrACS(year, "WP", ts_or_td))

obs_lifetimes_NA = np.array(obs_lifetimes_NA)
obs_lifetimes_NEP = np.array(obs_lifetimes_NEP)
obs_lifetimes_NWP = np.array(obs_lifetimes_NWP)

# get the distribution and PDF
shape, loc, scale = weibull_min.fit(obs_lifetimes_NA)
prob_obs_NA = weibull_min.pdf(values, shape, loc=loc, scale=scale)

shape, loc, scale = weibull_min.fit(obs_lifetimes_NEP)
prob_obs_NEP = weibull_min.pdf(values, shape, loc=loc, scale=scale)

shape, loc, scale = weibull_min.fit(obs_lifetimes_NWP)
prob_obs_NWP = weibull_min.pdf(values, shape, loc=loc, scale=scale)

obs_mo_count_NA_p1 = (
    obs_mo_count_NA_p1 / np.array(obs_TC_count_NA)[i_p1_start : i_p1_stop + 1].T
)
obs_mo_count_NEP_p1 = (
    obs_mo_count_NEP_p1 / np.array(obs_TC_count_NEP)[i_p1_start : i_p1_stop + 1].T
)
obs_mo_count_NWP_p1 = (
    obs_mo_count_NWP_p1 / np.array(obs_TC_count_NWP)[i_p1_start : i_p1_stop + 1].T
)

obs_mo_count_NA_p2 = (
    obs_mo_count_NA_p2 / np.array(obs_TC_count_NA)[i_p2_start : i_p2_stop + 1].T
)
obs_mo_count_NEP_p2 = (
    obs_mo_count_NEP_p2 / np.array(obs_TC_count_NEP)[i_p2_start : i_p2_stop + 1].T
)
obs_mo_count_NWP_p2 = (
    obs_mo_count_NWP_p2 / np.array(obs_TC_count_NWP)[i_p2_start : i_p2_stop + 1].T
)

In [ ]:
corr_TC_NA = np.corrcoef(f_ensmean_count_NA.flatten(), obs_TC_count_NA)[0, 1]
corr_HR_NA = np.corrcoef(f_ensmean_count_NA.flatten(), obs_HR_count_NA)[0, 1]

corr_TC_NEP = np.corrcoef(
    f_ensmean_count_NEP[9 : len(obs_TY_count_NEP)].flatten(),
    obs_TC_count_NEP[9 : len(obs_TY_count_NEP)],
)[0, 1]
corr_HR_NEP = np.corrcoef(
    f_ensmean_count_NEP[9 : len(obs_TY_count_NEP)].flatten(),
    obs_TY_count_NEP[9 : len(obs_TY_count_NEP)],
)[0, 1]

corr_TC_NWP = np.corrcoef(
    f_ensmean_count_NWP[5 : len(obs_TY_count_NWP)].flatten(),
    obs_TC_count_NWP[5 : len(obs_TY_count_NWP)],
)[0, 1]
corr_TY_NWP = np.corrcoef(
    f_ensmean_count_NWP[5 : len(obs_TY_count_NWP)].flatten(),
    obs_TY_count_NWP[5 : len(obs_TY_count_NWP)],
)[0, 1]

In [ ]:
# NA
f_lifetime_NA_p1 = []
f_lifetime_NA_p2 = []
obs_lifetime_NA_p1 = []
obs_lifetime_NA_p2 = []
f_lifetime_NA = []
obs_lifetime_NA = []

for iyr, yr in enumerate(range(start, stop + 1)):
    if yr >= p1_start and yr <= p1_stop:
        for iens, ens in enumerate(range(1, nens + 1)):
            if not any(iyr == (f_exclude[i][0] - start) and iens == (f_exclude[i][1] - 1) for i in range(len(f_exclude))):
                f_lifetime_NA_p1.extend(get_lifetime_NA(namelist_f[iyr][iens]))
        obs_lifetime_NA_p1.extend(get_lifetime_IBTrACS(yr, "AL", ts_or_td))

    elif yr >= p2_start and yr <= p2_stop:
        for iens, ens in enumerate(range(1, nens + 1)):
            if not any(iyr == (f_exclude[i][0] - start) and iens == (f_exclude[i][1] - 1) for i in range(len(f_exclude))):
                f_lifetime_NA_p2.extend(get_lifetime_NA(namelist_f[iyr][iens]))
        obs_lifetime_NA_p2.extend(get_lifetime_IBTrACS(yr, "AL", ts_or_td))

    for iens, ens in enumerate(range(1, nens + 1)):
        if not any(iyr == (f_exclude[i][0] - start) and iens == (f_exclude[i][1] - 1) for i in range(len(f_exclude))):
            f_lifetime_NA.extend(get_lifetime_NA(namelist_f[iyr][iens]))
    obs_lifetime_NA.extend(get_lifetime_IBTrACS(yr, "AL", ts_or_td))

statistic_NA, p_value_NA = ks_2samp(f_lifetime_NA_p1, f_lifetime_NA_p2)
statistic_obs_NA, p_value_obs_NA = ks_2samp(obs_lifetime_NA_p1, obs_lifetime_NA_p2)

# NWP
f_lifetime_NWP_p1 = []
f_lifetime_NWP_p2 = []
obs_lifetime_NWP_p1 = []
obs_lifetime_NWP_p2 = []
f_lifetime_NWP = []
obs_lifetime_NWP = []

for iyr, yr in enumerate(range(start, stop + 1)):
    if yr >= p1_start and yr <= p1_stop:
        for iens, ens in enumerate(range(1, nens + 1)):
            if not any(iyr == (f_exclude[i][0] - start) and iens == (f_exclude[i][1] - 1) for i in range(len(f_exclude))):
                f_lifetime_NWP_p1.extend(get_lifetime_NWP(namelist_f[iyr][iens]))
        obs_lifetime_NWP_p1.extend(get_lifetime_IBTrACS(yr, "WP", ts_or_td))

    elif yr >= p2_start and yr <= p2_stop:
        for iens, ens in enumerate(range(1, nens + 1)):
            if not any(iyr == (f_exclude[i][0] - start) and iens == (f_exclude[i][1] - 1) for i in range(len(f_exclude))):
                f_lifetime_NWP_p2.extend(get_lifetime_NWP(namelist_f[iyr][iens]))
        obs_lifetime_NWP_p2.extend(get_lifetime_IBTrACS(yr, "WP", ts_or_td))

    for iens, ens in enumerate(range(1, nens + 1)):
        if not any(iyr == (f_exclude[i][0] - start) and iens == (f_exclude[i][1] - 1) for i in range(len(f_exclude))):
            f_lifetime_NWP.extend(get_lifetime_NWP(namelist_f[iyr][iens]))
    obs_lifetime_NWP.extend(get_lifetime_IBTrACS(yr, "WP", ts_or_td))

statistic_NWP, p_value_NWP = ks_2samp(f_lifetime_NWP_p1, f_lifetime_NWP_p2)
statistic_obs_NWP, p_value_obs_NWP = ks_2samp(obs_lifetime_NWP_p1, obs_lifetime_NWP_p2)

# NEP
f_lifetime_NEP_p1 = []
f_lifetime_NEP_p2 = []
obs_lifetime_NEP_p1 = []
obs_lifetime_NEP_p2 = []
f_lifetime_NEP = []
obs_lifetime_NEP = []

for iyr, yr in enumerate(range(start, stop + 1)):
    if yr >= p1_start and yr <= p1_stop:
        for iens, ens in enumerate(range(1, nens + 1)):
            if not any(iyr == (f_exclude[i][0] - start) and iens == (f_exclude[i][1] - 1) for i in range(len(f_exclude))):
                f_lifetime_NEP_p1.extend(get_lifetime_NEP(namelist_f[iyr][iens]))
        obs_lifetime_NEP_p1.extend(get_lifetime_IBTrACS(yr, "EP", ts_or_td))

    elif yr >= p2_start and yr <= p2_stop:
        for iens, ens in enumerate(range(1, nens + 1)):
            if not any(iyr == (f_exclude[i][0] - start) and iens == (f_exclude[i][1] - 1) for i in range(len(f_exclude))):
                f_lifetime_NEP_p2.extend(get_lifetime_NEP(namelist_f[iyr][iens]))
        obs_lifetime_NEP_p2.extend(get_lifetime_IBTrACS(yr, "EP", ts_or_td))

    for iens, ens in enumerate(range(1, nens + 1)):
        if not any(iyr == (f_exclude[i][0] - start) and iens == (f_exclude[i][1] - 1) for i in range(len(f_exclude))):
            f_lifetime_NEP.extend(get_lifetime_NEP(namelist_f[iyr][iens]))
    obs_lifetime_NEP.extend(get_lifetime_IBTrACS(yr, "EP", ts_or_td))

statistic_NEP, p_value_NEP = ks_2samp(f_lifetime_NEP_p1, f_lifetime_NEP_p2)
statistic_obs_NEP, p_value_obs_NEP = ks_2samp(obs_lifetime_NEP_p1, obs_lifetime_NEP_p2)

bins = np.arange(0, 22, 1)

f_lifetime_NA_p1 = np.histogram(np.array(f_lifetime_NA_p1), bins=bins, density=True)[0]
f_lifetime_NEP_p1 = np.histogram(np.array(f_lifetime_NEP_p1), bins=bins, density=True)[0]
f_lifetime_NWP_p1 = np.histogram(np.array(f_lifetime_NWP_p1), bins=bins, density=True)[0]

f_lifetime_NA_p2 = np.histogram(np.array(f_lifetime_NA_p2), bins=bins, density=True)[0]
f_lifetime_NEP_p2 = np.histogram(np.array(f_lifetime_NEP_p2), bins=bins, density=True)[0]
f_lifetime_NWP_p2 = np.histogram(np.array(f_lifetime_NWP_p2), bins=bins, density=True)[0]

obs_lifetime_NA_p1 = np.histogram(np.array(obs_lifetime_NA_p1), bins=bins, density=True)[0]
obs_lifetime_NEP_p1 = np.histogram(np.array(obs_lifetime_NEP_p1), bins=bins, density=True)[0]
obs_lifetime_NWP_p1 = np.histogram(np.array(obs_lifetime_NWP_p1), bins=bins, density=True)[0]

obs_lifetime_NA_p2 = np.histogram(np.array(obs_lifetime_NA_p2), bins=bins, density=True)[0]
obs_lifetime_NEP_p2 = np.histogram(np.array(obs_lifetime_NEP_p2), bins=bins, density=True)[0]
obs_lifetime_NWP_p2 = np.histogram(np.array(obs_lifetime_NWP_p2), bins=bins, density=True)[0]

f_lifetime_NA_p2mp1 = f_lifetime_NA_p2 / np.sum(f_lifetime_NA_p2) - f_lifetime_NA_p1 / np.sum(f_lifetime_NA_p1)
f_lifetime_NEP_p2mp1 = f_lifetime_NEP_p2 / np.sum(f_lifetime_NEP_p2) - f_lifetime_NEP_p1 / np.sum(f_lifetime_NEP_p1)
f_lifetime_NWP_p2mp1 = f_lifetime_NWP_p2 / np.sum(f_lifetime_NWP_p2) - f_lifetime_NWP_p1 / np.sum(f_lifetime_NWP_p1)

obs_lifetime_NA_p2mp1 = obs_lifetime_NA_p2 / np.sum(obs_lifetime_NA_p2) - obs_lifetime_NA_p1 / np.sum(obs_lifetime_NA_p1)
obs_lifetime_NEP_p2mp1 = obs_lifetime_NEP_p2 / np.sum(obs_lifetime_NEP_p2) - obs_lifetime_NEP_p1 / np.sum(obs_lifetime_NEP_p1)
obs_lifetime_NWP_p2mp1 = obs_lifetime_NWP_p2 / np.sum(obs_lifetime_NWP_p2) - obs_lifetime_NWP_p1 / np.sum(obs_lifetime_NWP_p1)

In [ ]:
statistic_f_v_obs_NA, p_value_f_v_obs_NA = ks_2samp(np.random.choice(f_lifetime_NA,size=500), np.random.choice(obs_lifetime_NA,size=500))
statistic_f_v_obs_NEP, p_value_f_v_obs_NEP = ks_2samp(np.random.choice(f_lifetime_NEP,size=500), np.random.choice(obs_lifetime_NEP,size=500))
statistic_f_v_obs_NWP, p_value_f_v_obs_NWP = ks_2samp(np.random.choice(f_lifetime_NWP,size=500), np.random.choice(obs_lifetime_NWP,size=500))

print("NA p(F, Obs.) =",p_value_f_v_obs_NA)
print("NEP p(F, Obs.) =",p_value_f_v_obs_NEP)
print("NWP p(F, Obs.) =",p_value_f_v_obs_NWP)

In [ ]:
def add_trend_line(
    ax, x, y, color, label=None, linewidth=2, linestyle="-", zorder=None
):
    coeffs = np.polyfit(x, y, 1)
    trend = np.poly1d(coeffs)(x)
    ax.plot(
        x,
        trend,
        color=color,
        label=label,
        linewidth=linewidth,
        linestyle=linestyle,
        zorder=zorder,
    )

## Plot

In [ ]:
yrlist = np.arange(start, stop + 1)

fig, axs = plt.subplots(ncols=3, nrows=3, figsize=(8, 8))
axs = axs.flatten()

"""
TC COUNTS
"""

"""
NA
"""

ax = axs[0]

f_ensmean_count_NA = f_ensmean_count_NA.flatten()

# factual
ax.plot(
    yrlist,
    f_ensmean_count_NA,
    color="tab:orange",
    label="Factual",
    linewidth=0.8,
    zorder=4,
)
ax.fill_between(
    yrlist,
    f_twenty_NA.flatten(),
    f_eighty_NA.flatten(),
    color="tab:orange",
    alpha=0.25,
)

# # observed HRs
# ax.plot(
#     yrlist,
#     obs_HR_count_NA,
#     color="tab:blue",
#     linewidth=0.8,
#     label="Obs. HRs/TYs",
# )

# observed TCs
ax.plot(
    yrlist,
    obs_TC_count_NA,
    color="k",
    linewidth=0.8,
    label="Obs. TCs",
)

# 1949-2023 trend
mask_full = (yrlist >= 1949) & (yrlist <= 2023)
add_trend_line(
    ax,
    yrlist[mask_full],
    f_ensmean_count_NA[mask_full],
    color="tab:orange",
    linestyle="--",
    zorder=5,
)

# # 1949–1985 trend
# mask_p1 = (yrlist >= p1_start) & (yrlist <= p1_stop)
# add_trend_line(
#     ax,
#     yrlist[mask_p1],
#     f_ensmean_count_NA[mask_p1],
#     color="tab:orange",
#     label="Factual Trend",
#     linestyle="-",
#     zorder=5,
# )

# # 1987–2023 trend
# mask_p2 = (yrlist >= p2_start) & (yrlist <= p2_stop)
# add_trend_line(
#     ax,
#     yrlist[mask_p2],
#     f_ensmean_count_NA[mask_p2],
#     color="tab:orange",
#     linestyle="-",
#     zorder=5,
# )

# IBTrACS TC trend
# 1949–2023
add_trend_line(
    ax,
    yrlist[mask_full],
    np.array(obs_TC_count_NA)[mask_full],
    color="k",
    linestyle="--",
    zorder=6,
)

# # 1949–1985
# add_trend_line(
#     ax,
#     yrlist[mask_p1],
#     np.array(obs_TC_count_NA)[mask_p1],
#     color="tab:green",
#     label="Obs. TC Trend",
#     linestyle="-",
#     zorder=5,
# )

# # 1987–2023
# add_trend_line(
#     ax,
#     yrlist[mask_p2],
#     np.array(obs_TC_count_NA)[mask_p2],
#     color="tab:green",
#     linestyle="-",
#     zorder=5,
# )

# # IBTrACS HR trend
# # 1949–2023
# add_trend_line(
#     ax,
#     yrlist[mask_full],
#     np.array(obs_HR_count_NA)[mask_full],
#     color="indigo",
#     linestyle="--",
#     zorder=6,
# )

# # 1949–1985
# add_trend_line(
#     ax,
#     yrlist[mask_p1],
#     np.array(obs_HR_count_NA)[mask_p1],
#     color="tab:purple",
#     label="Obs. HR Trend",
#     linestyle="-",
#     zorder=5,
# )

# # 1987–2023
# add_trend_line(
#     ax,
#     yrlist[mask_p2],
#     np.array(obs_HR_count_NA)[mask_p2],
#     color="tab:purple",
#     linestyle="-",
#     zorder=5,
# )


# # correlation coefficient
# ax.text(
#     -2.35,
#     3.8,
#     f"cc(F, HRs)={corr_HR_NA:.2f}\ncc(F, TCs)={corr_TC_NA:.2f}",
#     transform=plt.gca().transAxes,
#     fontsize=10,
#     bbox=dict(facecolor="white", alpha=0.7, edgecolor="grey"),
# )

ax.set_ylim([0, 21])
# ax.set_xlabel("Year", fontsize=10)
ax.set_ylabel("Count", fontsize=10)
ax.set_xticks(np.arange(1940, 2020 + 20, 20))
ax.set_title("(a) NA TC Count", fontsize=14)
ax.tick_params(axis="both", which="major", labelsize=10)
ax.tick_params(axis="both", which="minor", labelsize=10)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(3.5, 0.75),
    fontsize=10,
    frameon=True,
)
ax.grid(alpha=0.3)

"""
NWP
"""

ax = axs[2]

f_ensmean_count_NWP = f_ensmean_count_NWP.flatten()

# factual
ax.plot(
    yrlist,
    f_ensmean_count_NWP,
    color="tab:orange",
    label="Factual",
    linewidth=0.8,
    zorder=4,
)
ax.fill_between(
    yrlist,
    f_twenty_NWP.flatten(),
    f_eighty_NWP.flatten(),
    color="tab:orange",
    alpha=0.25,
)


# # observed HRs
# ax.plot(
#     yrlist,
#     obs_TY_count_NWP,
#     color="tab:blue",
#     linewidth=0.8,
#     label="Obs. HRs/TYs",
# )

# observed TCs
ax.plot(
    yrlist,
    obs_TC_count_NWP,
    color="k",
    linewidth=0.8,
    label="Obs. TCs",
)

# 1949-2023 trend
add_trend_line(
    ax,
    yrlist[mask_full],
    f_ensmean_count_NWP[mask_full],
    color="tab:orange",
    linestyle="--",
    zorder=5,
)

# # 1949–1985 trend
# add_trend_line(
#     ax,
#     yrlist[mask_p1],
#     f_ensmean_count_NWP[mask_p1],
#     color="tab:orange",
#     linestyle="-",
#     zorder=5,
# )

# # 1987–2023 trend
# add_trend_line(
#     ax,
#     yrlist[mask_p2],
#     f_ensmean_count_NWP[mask_p2],
#     color="tab:orange",
#     linestyle="-",
#     zorder=5,
# )

# IBTrACS TC trend
# 1949–2023
add_trend_line(
    ax,
    yrlist[mask_full],
    np.array(obs_TC_count_NWP)[mask_full],
    color="k",
    linestyle="--",
    zorder=6,
)

# # 1949–1985
# add_trend_line(
#     ax,
#     yrlist[mask_p1],
#     np.array(obs_TC_count_NWP)[mask_p1],
#     color="tab:green",
#     linestyle="-",
#     zorder=5,
# )

# # 1987–2023
# add_trend_line(
#     ax,
#     yrlist[mask_p2],
#     np.array(obs_TC_count_NWP)[mask_p2],
#     color="tab:green",
#     linestyle="-",
#     zorder=5,
# )

# # IBTrACS HR trend
# # 1949–2023
# add_trend_line(
#     ax,
#     yrlist[mask_full],
#     np.array(obs_TY_count_NWP)[mask_full],
#     color="indigo",
#     linestyle="--",
#     zorder=6,
# )

# # 1949–1985
# add_trend_line(
#     ax,
#     yrlist[mask_p1],
#     np.array(obs_TY_count_NWP)[mask_p1],
#     color="tab:purple",
#     linestyle="-",
#     zorder=5,
# )

# # 1987–2023
# add_trend_line(
#     ax,
#     yrlist[mask_p2],
#     np.array(obs_TY_count_NWP)[mask_p2],
#     color="tab:purple",
#     linestyle="-",
#     zorder=5,
# )

# # correlation coefficient
# ax.text(
#     -1.15,
#     3.8,
#     f"cc(F, TYs)={corr_TY_NWP:.2f}\ncc(F, TCs)={corr_TC_NWP:.2f}",
#     transform=plt.gca().transAxes,
#     fontsize=10,
#     bbox=dict(facecolor="white", alpha=0.7, edgecolor="grey"),
# )

ax.set_ylim([0, 32])
# ax.set_xlabel("Year", fontsize=10)
ax.set_xticks(np.arange(1940, 2020 + 20, 20))
ax.tick_params(axis="both", which="major", labelsize=10)
ax.tick_params(axis="both", which="minor", labelsize=10)
ax.set_title("(c) NWP TC Count", fontsize=14)
ax.grid(alpha=0.3)

"""
NEP
"""

ax = axs[1]

f_ensmean_count_NEP = f_ensmean_count_NEP.flatten()

# factual
ax.plot(
    yrlist,
    f_ensmean_count_NEP,
    color="tab:orange",
    label="Factual",
    linewidth=0.8,
    zorder=4,
)
ax.fill_between(
    yrlist,
    f_twenty_NEP.flatten(),
    f_eighty_NEP.flatten(),
    color="tab:orange",
    alpha=0.25,
)

# # observed HRs
# ax.plot(
#     yrlist,
#     obs_TY_count_NEP,
#     color="tab:blue",
#     linewidth=0.8,
#     label="Obs. HRs/TYs",
# )

# observed TCs
ax.plot(
    yrlist,
    obs_TC_count_NEP,
    color="k",
    linewidth=0.8,
    label="Obs. TCs",
)

# 1949-2023 trend
mask_full = (yrlist >= 1949) & (yrlist <= 2023)
add_trend_line(
    ax,
    yrlist[mask_full],
    f_ensmean_count_NEP[mask_full],
    color="tab:orange",
    linestyle="--",
    zorder=5,
)

# # 1949–1985 trend
# mask_p1 = (yrlist >= p1_start) & (yrlist <= p1_stop)
# add_trend_line(
#     ax,
#     yrlist[mask_p1],
#     f_ensmean_count_NEP[mask_p1],
#     color="tab:orange",
#     linestyle="-",
#     zorder=5,
# )

# # 1987–2023 trend
# add_trend_line(
#     ax,
#     yrlist[mask_p2],
#     f_ensmean_count_NEP[mask_p2],
#     color="tab:orange",
#     linestyle="-",
#     zorder=5,
# )

# IBTrACS TC trend
# 1949–2023
add_trend_line(
    ax,
    yrlist[mask_full],
    np.array(obs_TC_count_NEP)[mask_full],
    color="k",
    linestyle="--",
    zorder=6,
)

# # 1949–1985
# add_trend_line(
#     ax,
#     yrlist[mask_p1],
#     np.array(obs_TC_count_NEP)[mask_p1],
#     color="tab:green",
#     linestyle="-",
#     zorder=5,
# )

# # 1987–2023
# add_trend_line(
#     ax,
#     yrlist[mask_p2],
#     np.array(obs_TC_count_NEP)[mask_p2],
#     color="tab:green",
#     linestyle="-",
#     zorder=5,
# )

# # IBTrACS TC trend
# # 1949–2023
# add_trend_line(
#     ax,
#     yrlist[mask_full],
#     np.array(obs_TY_count_NEP)[mask_full],
#     color="indigo",
#     linestyle="--",
#     zorder=6,
# )

# # 1949–1985
# mask_p1 = (yrlist >= p1_start) & (yrlist <= p1_stop)
# add_trend_line(
#     ax,
#     yrlist[mask_p1],
#     np.array(obs_TY_count_NEP)[mask_p1],
#     color="tab:purple",
#     linestyle="-",
#     zorder=5,
# )

# # 1987–2023
# add_trend_line(
#     ax,
#     yrlist[mask_p2],
#     np.array(obs_TY_count_NEP)[mask_p2],
#     color="tab:purple",
#     linestyle="-",
#     zorder=5,
# )

# # correlation coefficient
# ax.text(
#     0.05,
#     3.8,
#     f"cc(F, HRs)={corr_HR_NEP:.2f}\ncc(F, TCs)={corr_TC_NEP:.2f}",
#     transform=plt.gca().transAxes,
#     fontsize=10,
#     bbox=dict(facecolor="white", alpha=0.7, edgecolor="grey"),
# )

ax.set_ylim([0, 24])
ax.set_xticks(np.arange(1940, 2020 + 20, 20))
ax.tick_params(axis="both", which="major", labelsize=10)
ax.tick_params(axis="both", which="minor", labelsize=10)
ax.set_title("(b) NEP TC Count", fontsize=14)
ax.grid(alpha=0.3)

"""
TC LIFETIMES
"""

"""
NA
"""

ax = axs[3]

ax.bar(
    np.arange(0, 21, 1),
    obs_lifetime_NA_p2mp1,
    align="edge",
    edgecolor="dimgray",
    color="dimgray",
    alpha=0.5,
    label="Obs. P2-P1",
)
ax.bar(
    np.arange(0, 21, 1),
    obs_lifetime_NA_p2mp1,
    align="edge",
    edgecolor="dimgray",
    facecolor="none",
)

# plot factual
ax.bar(
    np.arange(0, 21, 1),
    f_lifetime_NA_p2mp1,
    align="edge",
    edgecolor="gold",
    color="gold",
    alpha=0.5,
    label="Factual P2-P1",
)
ax.bar(
    np.arange(0, 21, 1),
    f_lifetime_NA_p2mp1,
    align="edge",
    edgecolor="gold",
    facecolor="none",
)

ax.hist(
    obs_lifetime_NA,
    bins=np.arange(0, 22, 1),
    density=True,
    color="k",
    histtype="step",
    linewidth=1.5,
    label="Obs.",
)
ax.hist(
    f_lifetime_NA,
    bins=np.arange(0, 22, 1),
    density=True,
    color="tab:orange",
    histtype="step",
    linewidth=1.5,
    label="Factual",
)

# arrange figure
ax.legend(
    loc="upper left",
    bbox_to_anchor=(3.5, 0.75),
    fontsize=10,
    frameon=True,
)

# p value
ax.text(
    -1.435,
    2.3,
    f"p(Obs. P1, Obs. P2)={p_value_obs_NA:.2f}\np(F P1, F P2)={p_value_NA:.2f}",
    transform=plt.gca().transAxes,
    horizontalalignment="right",
    fontsize=10,
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="grey"),
)

ax.grid(alpha=0.3)
ax.set_xlim(values[0], values[-1])
ax.set_ylim(-0.06, 0.21)
ax.set_xlabel("Lifetime (days)", fontsize=10)
ax.set_ylabel("Density (normalized)", fontsize=10)
ax.set_title("(d) NA Lifetimes", fontsize=14)
ax.tick_params(axis="both", which="major", labelsize=10)
ax.tick_params(axis="both", which="minor", labelsize=10)

"""
NWP
"""

ax = axs[5]

ax.bar(
    np.arange(0, 21, 1),
    obs_lifetime_NWP_p2mp1,
    align="edge",
    edgecolor="dimgray",
    color="dimgray",
    alpha=0.5,
)
ax.bar(
    np.arange(0, 21, 1),
    obs_lifetime_NWP_p2mp1,
    align="edge",
    edgecolor="dimgray",
    facecolor="none",
)

# plot factual
ax.bar(
    np.arange(0, 21, 1),
    f_lifetime_NWP_p2mp1,
    align="edge",
    edgecolor="gold",
    color="gold",
    alpha=0.5,
)
ax.bar(
    np.arange(0, 21, 1),
    f_lifetime_NWP_p2mp1,
    align="edge",
    edgecolor="gold",
    facecolor="none",
)

ax.hist(
    obs_lifetime_NWP,
    bins=np.arange(0, 22, 1),
    density=True,
    color="k",
    histtype="step",
    linewidth=1.5,
)
ax.hist(
    f_lifetime_NWP,
    bins=np.arange(0, 22, 1),
    density=True,
    color="tab:orange",
    histtype="step",
    linewidth=1.5,
)

# p value
ax.text(
    .965,
    2.3,
    f"p(Obs. P1, Obs. P2)={p_value_obs_NWP:.2f}\np(F P1, F P2)={p_value_NWP:.2f}",
    transform=plt.gca().transAxes,
    horizontalalignment="right",
    fontsize=10,
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="grey"),
)

# arrange figure
ax.grid(alpha=0.3)
ax.set_xlim(values[0], values[-1])
ax.set_ylim(-0.06, 0.21)
ax.set_xlabel("Lifetime (days)", fontsize=10)
ax.set_title("(f) NWP Lifetimes", fontsize=14)
ax.tick_params(axis="both", which="major", labelsize=10)
ax.tick_params(axis="both", which="minor", labelsize=10)


"""
NEP
"""
ax = axs[4]


ax.bar(
    np.arange(0, 21, 1),
    obs_lifetime_NEP_p2mp1,
    align="edge",
    edgecolor="dimgray",
    color="dimgray",
    alpha=0.5,
)
ax.bar(
    np.arange(0, 21, 1),
    obs_lifetime_NEP_p2mp1,
    align="edge",
    edgecolor="dimgray",
    facecolor="none",
)

# plot factual
ax.bar(
    np.arange(0, 21, 1),
    f_lifetime_NEP_p2mp1,
    align="edge",
    edgecolor="gold",
    color="gold",
    alpha=0.5,
)
ax.bar(
    np.arange(0, 21, 1),
    f_lifetime_NEP_p2mp1,
    align="edge",
    edgecolor="gold",
    facecolor="none",
)

ax.hist(
    obs_lifetime_NEP,
    bins=np.arange(0, 22, 1),
    density=True,
    color="k",
    histtype="step",
    linewidth=1.5,
)
ax.hist(
    f_lifetime_NEP,
    bins=np.arange(0, 22, 1),
    density=True,
    color="tab:orange",
    histtype="step",
    linewidth=1.5,
)

# p value
ax.text(
    -.225,
    2.3,
    f"p(Obs. P1, Obs. P2)={p_value_obs_NEP:.2f}\np(F P1, F P2)={p_value_NEP:.2f}",
    transform=plt.gca().transAxes,
    horizontalalignment="right",
    fontsize=10,
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="grey"),
)

# arrange figure
ax.set_xlim(values[0], values[-1])
ax.set_ylim(-0.06, 0.26)
ax.set_xlabel("Lifetime (days)", fontsize=10)
ax.set_title("(e) NEP Lifetimes", fontsize=14)
ax.tick_params(axis="both", which="major", labelsize=10)
ax.tick_params(axis="both", which="minor", labelsize=10)
ax.grid(alpha=0.3)


"""
Seasonality
"""

months = ["Jul", "Aug", "Sept", "Oct"]

positions = []
group_spacing = 3  # controls spacing between month groups
intra_group_spacing = 0.6  # spacing between boxes in a pair
inter_group_spacing = 1.5

for i in range(len(months)):
    start_spacing = i * group_spacing
    positions.extend(
        [
            start_spacing,
            start_spacing + intra_group_spacing,
            start_spacing + inter_group_spacing,
            start_spacing + inter_group_spacing + intra_group_spacing,
        ]
    )

# Use midpoints between first and last box in each month group for x-ticks
mid_positions = [
    (positions[i] + positions[i + 3]) / 2 for i in range(0, len(positions), 4)
]

ax = axs[6]

data = []
for i in range(len(months)):
    data.extend(
        [
            mo_count_NA_p1[i],
            mo_count_NA_p2[i],
            obs_mo_count_NA_p1[i],
            obs_mo_count_NA_p2[i],
        ]
    )


boxes = ax.boxplot(
    data,
    positions=positions,
    patch_artist=True,
    medianprops=dict(color="tab:blue", linewidth=2),
    showfliers=False,
)

for i, box in enumerate(boxes["boxes"]):
    box.set_linewidth(1.25)
    if i % 4 == 0:
        box.set_facecolor("tab:green")
    elif i % 4 == 1:
        box.set_facecolor("tab:orange")
    elif i % 4 == 2:
        box.set_facecolor("none")
        box.set_edgecolor("dimgray")
        # box.set_linestyle("dashed")
    elif i % 4 == 3:
        box.set_facecolor("none")
        # box.set_linestyle("dotted")

ax.set_xticks(mid_positions)
ax.set_xticklabels(months, fontsize=10)
ax.set_yticks([0, 0.2, 0.4, 0.6])
ax.set_yticklabels([0, 0.2, 0.4, 0.6], fontsize=10)
ax.set_ylim(0, 0.75)
ax.set_ylabel("Count", fontsize=10)
ax.set_title("(g) NA Monthly Count", fontsize=14)
ax.grid(alpha=0.3)


"""
NWP
"""
ax = axs[8]

data = []
for i in range(len(months)):
    data.extend(
        [
            mo_count_NWP_p1[i],
            mo_count_NWP_p2[i],
            obs_mo_count_NWP_p1[i],
            obs_mo_count_NWP_p2[i],
        ]
    )  # now three per month


boxes = ax.boxplot(
    data,
    positions=positions,
    patch_artist=True,
    medianprops=dict(color="tab:blue", linewidth=2),
    showfliers=False,
)

for i, box in enumerate(boxes["boxes"]):
    box.set_linewidth(1.25)
    if i % 4 == 0:
        box.set_facecolor("tab:green")
    elif i % 4 == 1:
        box.set_facecolor("tab:orange")
    elif i % 4 == 2:
        box.set_facecolor("none")
        box.set_edgecolor("dimgray")
        # box.set_linestyle("dashed")
    elif i % 4 == 3:
        box.set_facecolor("none")
        # box.set_linestyle("dotted")

ax.set_xticks(mid_positions)
ax.set_xticklabels(months, fontsize=10)
ax.set_yticks([0, 0.2, 0.4, 0.6])
ax.set_yticklabels([0, 0.2, 0.4, 0.6], fontsize=10)
ax.set_ylim(0, 0.75)
ax.set_title("(i) NWP Monthly Count", fontsize=14)
ax.grid(alpha=0.3)

"""
NEP
"""
ax = axs[7]

data = []
for i in range(len(months)):
    data.extend(
        [
            mo_count_NEP_p1[i],
            mo_count_NEP_p2[i],
            obs_mo_count_NEP_p1[i],
            obs_mo_count_NEP_p2[i],
        ]
    )  # now three per month

boxes = ax.boxplot(
    data,
    positions=positions,
    patch_artist=True,
    medianprops=dict(color="tab:blue", linewidth=2),
    showfliers=False,
)

for i, box in enumerate(boxes["boxes"]):
    box.set_linewidth(1.25)
    if i % 4 == 0:
        box.set_facecolor("tab:green")
    elif i % 4 == 1:
        box.set_facecolor("tab:orange")
    elif i % 4 == 2:
        box.set_facecolor("none")
        box.set_edgecolor("dimgray")
        # box.set_linestyle("dashed")
    elif i % 4 == 3:
        box.set_facecolor("none")
        # box.set_linestyle("dotted")

ax.set_xticks(mid_positions)
ax.set_xticklabels(months, fontsize=10)
ax.set_yticks([0, 0.2, 0.4, 0.6])
ax.set_yticklabels([0, 0.2, 0.4, 0.6], fontsize=10)
ax.set_ylim(0, 0.75)
ax.set_title("(h) NEP Monthly Count", fontsize=14)
ax.grid(alpha=0.3)

orange_patch = mpatches.Patch(
    facecolor="tab:orange", edgecolor="black", label="Factual P2"
)
green_patch = mpatches.Patch(
    facecolor="tab:green", edgecolor="black", label="Factual P1"
)

black_line = mpatches.Patch(
    facecolor="none",
    edgecolor="black",
    label="Obs. P2",
)
gray_line = mpatches.Patch(
    facecolor="none",
    edgecolor="dimgray",
    label="Obs. P1",
)

# Add legend to current axes
ax.legend(
    handles=[green_patch, orange_patch, gray_line, black_line],
    loc="upper left",
    bbox_to_anchor=(2.3, .8),
    fontsize=10,
    frameon=True,
)

plt.subplots_adjust(left=0, bottom=0, right=1, top=0.875, wspace=0.2, hspace=0.5)
plt.savefig("./figs/figure4.png", dpi=600, bbox_inches="tight")
plt.savefig("./figs/figure4.pdf", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
f_lifetime_NEP_p2mp1

In [ ]:
f_lifetime_NEP_p1

In [ ]:
f_lifetime_NEP